# QSS 20 Final Project — Data Pull & Merge

**Author:** Taj Horowitz  
**Updated:** May 2026

**Purpose:** Loads and cleans ACS state-level data, pulls NAEP math scores from the  
Nations Report Card API, pivots to wide format, merges the two datasets, and saves  
`data/merged_panel.csv` for use by the visualization notebook.

## Notebook roadmap
1. Imports  
2. Load and clean ACS data  
3. Pull NAEP scores from the Data Service API  
4. Pivot NAEP long → wide and merge onto ACS  
5. Inspect merged panel  
6. Save merged panel to CSV

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import re

## 2. Load and clean ACS data

Loads `acs_wmath.csv` (50 states × ~90 ACS columns) and does three things: drops irrelevant columns (math scores (will be using NAEP instead), income-allocation mechanics, granular household subtypes), renames columns to readable short prefixes (`pov_ratio_*`, `educ_*`, `hh_*`), and consolidates educational attainment into 5 standard buckets.

In [ ]:
# Data source: QSS 20 course repository
# https://github.com/herbertfreeze/QSS20-S26/tree/main/public_data
df = pd.read_csv("https://raw.githubusercontent.com/herbertfreeze/QSS20-S26/main/public_data/acs_wmath.csv")

# Drop columns that aren't useful for this analysis.
# Math scores come from NAEP instead, so the ACS math columns are redundant.
math_cols = [c for c in df.columns if c.startswith('math_')]

# Income allocation columns describe survey reporting mechanics, not actual income levels.
alloc_cols = [c for c in df.columns if 'allocation_of_household_income' in c]

# The ACS breaks down household composition into very granular subcategories
# (e.g. "biological child", "foster child", "roomer or boarder") that aren't
# meaningful at the state level for this analysis. Keep only the top-level household totals.
hh_drop_keywords = [
    'brother_or_sister', 'adopted_child', 'biological_child', 'stepchild',
    'grandchild', 'parentinlaw', 'soninlaw', 'foster_child',
    'housemate_or_roommate', 'roomer_or_boarder', 'unmarried_partner',
    'other_nonrelatives', 'other_relatives', 'not_living_alone'
]
hh_cols = [c for c in df.columns
           if 'household_type' in c
           and any(kw in c for kw in hh_drop_keywords)]

cols_to_drop = math_cols + alloc_cols + hh_cols
df = df.drop(columns=cols_to_drop)

# Rename columns to something readable.
# The raw ACS column names are long Census-style strings like
# "estimatetotaleducational_attainmentbachelors_degree"
# prefixes it so columns group nicely.
def clean_acs_col(col):
    if col in ('abbrev', 'FIPS', 'state'):
        return col
    if 'median_household_income' in col:
        return 'median_household_income'
    # For poverty ratio columns, pull out the income bracket from the name
    # using a regex. re.search finds the first match anywhere in the string,
    # and group(1) returns just the captured bracket (e.g. "under 50").
    if 'ratio_of_income_to_poverty' in col:
        match = re.search(r'(under \d+|\d+ to \d+|\d+ and over)', col)
        if match:
            bracket = match.group(1).replace(' to ', '_').replace(' ', '_')
            return f'pov_ratio_{bracket}'
    # For education and household columns, strip everything up to "estimatetotal"
    # (a Census artifact) and clean what remains into a snake_case label.
    if 'educational_attainment' in col:
        level = re.sub(r".*estimatetotal", "", col).strip()
        level = re.sub(r"[^a-zA-Z0-9]", "_", level).strip("_").lower()
        return f'educ_{level}'
    if 'household_type' in col:
        label = re.sub(r".*estimatetotal", "", col).strip()
        label = re.sub(r"[^a-zA-Z0-9]", "_", label).strip("_").lower()
        return f'hh_{label}'
    return col

df.columns = [clean_acs_col(c) for c in df.columns]

# The ACS reports educational attainment at a very fine grain
# for every grade level from "no schooling" through "doctorate". That's too
# granular to be useful as predictors, so collapse them into 5 standard buckets
# that match how education is typically categorized in social science research.
less_than_hs_cols = [c for c in df.columns if c.startswith('educ_') and
                     any(x in c for x in ['no_schooling', 'nursery', 'kindergarten',
                                           '1st', '2nd', '3rd', '4th', '5th',
                                           '6th', '7th', '8th', '9th', '10th',
                                           '11th', '12th_grade_no_diploma'])]
hs_cols = [c for c in df.columns if c.startswith('educ_') and
                  any(x in c for x in ['regular_high_school', 'ged'])]
some_col_cols = [c for c in df.columns if c.startswith('educ_') and
                  any(x in c for x in ['some_college', "associate"])]
bachelors_cols = [c for c in df.columns if c.startswith('educ_') and "bachelor" in c]
grad_cols = [c for c in df.columns if c.startswith('educ_') and
                  any(x in c for x in ["master", "professional", "doctorate"])]

df['educ_less_than_hs'] = df[less_than_hs_cols].sum(axis=1)
df['educ_hs_or_ged']    = df[hs_cols].sum(axis=1)
df['educ_some_college'] = df[some_col_cols].sum(axis=1)
df['educ_bachelors']    = df[bachelors_cols].sum(axis=1)
df['educ_graduate']     = df[grad_cols].sum(axis=1)

individual_educ_cols = less_than_hs_cols + hs_cols + some_col_cols + bachelors_cols + grad_cols
df = df.drop(columns=individual_educ_cols)

print(f"\nFinal ACS shape: {df.shape[0]} states x {df.shape[1]} columns")
df.head()

## 3. Pull NAEP scores from the Data Service API

Pulls mean math scores for 5 demographic breakdowns across grades 4 & 8 and years 2013–2019.

| Variable | Meaning |
|---|---|
| `SLUNCH3` | Free/reduced-price lunch eligibility |
| `GENDER`  | Male / Female |
| `SRACE10` | Race/ethnicity |
| `PARED`   | Parental education (grade 8 only) |
| `SCHTYPE` | Public vs. private school |

**All 50 states are passed in one request per variable/grade/year** (40 requests total instead of 2,000).  
Result: ~7,800 rows in long format. Saved to `data/naep_pulled.csv`.

In [ ]:
SUBSCALE  = 'MRPCM'   # math composite score 
VARIABLES = ['SLUNCH3', 'SRACE10', 'GENDER', 'PARED', 'SCHTYPE']
GRADES    = [4, 8]           # NAEP tests grades 4 and 8 on a consistent schedule; grade 12 data is sparse
YEARS     = [2013, 2015, 2017, 2019]  # administered every other year
STATES    = list(df['abbrev'].unique())

# Passing all states as a comma-separated string lets the API return all 50 at once.
# Without this, the loop would make ~1,800 requests; this way it's around 36.
ALL_STATES_STR = ','.join(STATES)


def pull_naep_all_states(variable, grade, year, states_str, retries=3):
    # The NAEP Data Service API works by building a URL with query parameters —
    # each &key=value is a filter the server applies before returning results.
    # The URL is constructed by string concatenation here because Python's adjacent
    # string literals auto-join, and adding variables mid-string requires the + operator.
    for attempt in range(retries):
        query = (
            'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?'
            'type=data'
            '&subject=mathematics'
            '&grade=' + str(grade) +
            '&subscale=' + SUBSCALE +
            '&variable=' + variable +
            '&jurisdiction=' + states_str +    # comma-separated state abbreviations
            '&stattype=MN:MN'                  # MN:MN requests mean scores, not percentiles
            '&Year=' + str(year)
        )

        try:
            # requests.get() fires a standard HTTP GET request. timeout=30 prevents
            # the script from hanging indefinitely if the server stops responding.
            r = requests.get(query, timeout=30)

            # The API response is JSON 
            # .json() deserializes it into an actual dict so the data can be accessed by key.
            data = r.json()

            # Using .get('result') instead of ['result'] means a missing key returns None
            # rather than raising a KeyError
            # safer when the API occasionally returns an error response instead of data.
            if data.get('result'):
                return pd.DataFrame(data['result'])

        except Exception as e:
            print(f'  Bulk attempt {attempt+1} failed: {e}')
            # sleep 1s on the first retry, 2s on the second, 4s on the third.
            
            time.sleep(2 ** attempt)

    # If the bulk request failed after all retries, fall back to requesting each state individually 
    print(f'  Falling back to per-state for {variable} grade {grade} {year}...')
    results_list = []

    for state in states_str.split(','):
        q = (
            'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?'
            'type=data'
            '&subject=mathematics'
            '&grade=' + str(grade) +
            '&subscale=' + SUBSCALE +
            '&variable=' + variable +
            '&jurisdiction=' + state +
            '&stattype=MN:MN'
            '&Year=' + str(year)
        )
        try:
            r = requests.get(q, timeout=15)
            data = r.json()
            if data.get('result'):
                results_list.append(pd.DataFrame(data['result']))
        except Exception as e:
            print(f'    Failed: {state} — {e}')

        time.sleep(0.3)  # brief pause to avoid triggering rate limiting

    # ignore_index=True resets row numbers after stacking — without it,
    # each per-state DataFrame would restart its index from 0, creating duplicates.
    if results_list:
        return pd.concat(results_list, ignore_index=True)

    return pd.DataFrame()


# PARED (parental education) is only collected at grade 8. Grade 4 students are
# too young to reliably report their parents' education level, so NAEP doesn't ask.
# Skipping this combination avoids ~200 requests that would return nothing.
GRADE_RESTRICTIONS = {
    'PARED': [8],
}

all_dfs = []

for variable in VARIABLES:
    # dict.get(key, default) returns the default value when the key isn't present,
    # so variables without restrictions automatically get both grades.
    allowed_grades = GRADE_RESTRICTIONS.get(variable, GRADES)

    for grade in allowed_grades:
        for year in YEARS:
            print(f'Pulling {variable}, grade {grade}, {year}...')
            result = pull_naep_all_states(variable, grade, year, ALL_STATES_STR)
            if not result.empty:
                all_dfs.append(result)
            time.sleep(0.5)

naep_raw = pd.concat(all_dfs, ignore_index=True)
print('\nTotal rows pulled:', len(naep_raw))

# The raw API response contains many columns — keep only the ones needed for analysis.
naep_clean = naep_raw[[
    'jurisdiction',    # state abbreviation (e.g. 'NH')
    'jurisLabel',      # full state name (e.g. 'New Hampshire')
    'grade',
    'year',
    'variable',        # demographic breakdown label (e.g. 'GENDER')
    'varValueLabel',   # category within that variable (e.g. 'Male', 'Female')
    'value'            # mean math score
]].copy()

naep_clean.columns = [
    'state_abbrev', 'state_name', 'grade', 'year',
    'variable', 'category', 'mean_score'
]

naep_clean['mean_score'] = naep_clean['mean_score'].round(2)

# NAEP suppresses scores when a subgroup has too few students to report reliably.
# Those show up as NaN after parsing — drop them rather than carry forward missing data.
naep_clean = naep_clean.dropna(subset=['mean_score'])

print('naep_clean shape:', naep_clean.shape)
naep_clean.head(15)

os.makedirs("data", exist_ok=True)
naep_clean.to_csv("data/naep_pulled.csv", index=False)
print("Saved to data/naep_pulled.csv")

## 4. Pivot NAEP wide and merge onto ACS

Pivots NAEP from long to wide (one row per state × grade × year, one column per subgroup score), replaces NAEP's `999` suppression sentinel with `NaN`, then left-merges the ACS onto the NAEP panel.

In [ ]:
# The NAEP data comes back in "long" format — one row per state per subgroup category.
# For example, NH grade 4 2019 appears as two separate rows: one for Male, one for Female.
# pivot_table() reshapes this so each state/grade/year combination is a single row,
# with separate columns for each subgroup score (gender_male, gender_female, etc.).
naep_wide = naep_clean.pivot_table(
    index=['state_abbrev', 'grade', 'year'],
    columns=['variable', 'category'],
    values='mean_score'
)

# pivot_table() creates a MultiIndex column like ('GENDER', 'Male').
# This flattens those into a single readable string like 'gender_male'.
naep_wide.columns = [
    (var + '_' + cat).lower().replace(' ', '_').replace("'", '').replace('/', '_')
    for var, cat in naep_wide.columns
]

naep_wide = naep_wide.reset_index()

# NAEP uses 999 as a sentinel value for suppressed scores (not actual data).
# Replace with pd.NA so these don't skew any calculations downstream.
naep_wide = naep_wide.replace(999, pd.NA)

print('NAEP wide shape:', naep_wide.shape)

# Merge ACS onto NAEP using a left join, keeping all NAEP rows.
# The ACS snapshot is from 2018, so it's the same for every year of NAEP data —
# this is a known limitation noted in the analysis.
merged = naep_wide.merge(df, left_on='state_abbrev', right_on='abbrev', how='left')
merged = merged.drop(columns=['abbrev'])  # redundant after the merge

print('Merged shape:', merged.shape)
print('Expected: ~400 rows (50 states x 2 grades x 4 years)')

# Sanity check — any state that failed to match will have NaN for income.
unmatched = merged[merged['median_household_income'].isna()]['state_abbrev'].tolist()
print('States with no ACS match:', unmatched)
merged.head(10)

## 5. Inspect merged panel

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
merged

## 6. Save merged panel to CSV

In [ ]:
os.makedirs("data", exist_ok=True)
merged.to_csv("data/merged_panel.csv", index=False)
print("Saved to data/merged_panel.csv —", merged.shape)
print("\nReady for 02_visualizations.ipynb")